# Pull selection and synthetic time variations

26 November 2026 session, continued after midnight on 27 November.

Notes only. Code is in src/synthetic.py and src/eda.py. Event definitions are in data/pull_windows.csv.

## Measured coverage

The processed dataset is concentrated around idle and light load. Only night.csv contains MAP above 100 kPa and TPS >= 80%; it has 164 rows meeting the TPS condition. This condition was used to inspect and improve coverage.

Synthetic data is intended to add different driving scenarios as well as defined risk variations. Both ordinary and risk scenarios need synthetic examples so that synthetic origin alone does not identify the class.

## Selected pulls

RPM, MAP and TPS plots were reviewed together, followed by manual checks in MegaLogViewer.

| Event | Start (s) | End (s) | Rows | Max RPM | Max MAP (kPa) | Max TPS (%) | Split |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| pull_1 | 225.698 | 255.209 | 415 | 3809 | 62 | 16 | development |
| pull_2 | 390.679 | 432.047 | 559 | 4140 | 144 | 47 | development |
| pull_3 | 454.832 | 493.993 | 527 | 5705 | 171 | 100 | development |
| pull_4 | 529.049 | 559.564 | 389 | 5476 | 175 | 100 | test |
| pull_5 | 564.647 | 591.978 | 344 | 3287 | 145.5 | 47 | development |

Pull 1 provides a lower-load example, pulls 2 and 5 include partial throttle and positive boost, and pulls 3 and 4 include full throttle. Event windows include surrounding transitions rather than only the peak values.

The selection changes only the synthetic template list. Other driving data remains in the processed logs.

## Grouping and evaluation plan

Each source pull and all its variations form one group. Assigning a variation to a different split from its source could leak the same event structure into evaluation.

pull_4 is reserved for the final test because it includes full throttle and the highest measured MAP. pull_3 leaves another full-throttle template in development. The split was chosen from operating coverage.

Development contains pulls 1, 2, 3 and 5. Event-wise cross-validation was decided: hold out one development event and its variations for validation, train on the other groups, then rotate.

Chapter 7, printed PDF pages 6–7 and section 11.1, gives the training/validation/test and cross-validation principles. Grouping by event adapts those principles to dependent log rows.

Before model fitting, the split for the remaining logs and the rest of night.csv must also be defined. Reserved event rows must not re-enter development through the full night.csv file.

## Time transformation

stretch_pull(base_pull, time_scale) copies the selected event and replaces Time with elapsed time multiplied by a positive scale factor:

$$t'_i = a(t_i-t_0).$$

Sensor values and row count stay unchanged. Every sensor uses the same new timestamps. RPM rate is recalculated from the modified time intervals within each segment.

$$r'_i = r_i/a$$

Here the rate relation applies to rates calculated from the same event rows. The first rate in each extracted segment is missing. The existing recorded rpm/s column is copied unchanged and is not a valid updated rate feature.

For pull_3, the original duration is 39.161 s. A factor of 1.10 gives 43.077 s. A factor of 2.00 gives 78.322 s and was used to make the effect clear in the plot. This is horizontal stretching, not a constant time shift: an interval from 20 to 25 s becomes 22 to 27.5 s at 1.10.

The transformation was moved into a function, then applied with nested loops over other events.

## Generated files and checks

The current experimental factors are 0.90, 0.95, 1.05, 1.10 and 1.15. Four development pulls and five factors produce 20 expected CSV files in data/synthetic.

Metadata fields are source_log, source_event, split, variant_id and time_scale. They record lineage and settings and are not model inputs. File names include the source event and factor, such as pull_3_stretch_1.10.csv.

Plots compared the original and transformed RPM, MAP and TPS on a shared elapsed-time axis.

## Assumptions and limits

Uniform time scaling preserves the measured curve shapes and their relative timing. It also stretches measurement fluctuations and sensor delays. It is not a physical engine simulation and does not model how changing acceleration would alter the other readings.

The selected factors are experimental settings, not scientifically established limits for this engine. Their usefulness still needs an unaugmented-versus-augmented comparison. Only rate and other time-dependent features change under this transformation; instantaneous sensor rows do not gain new amplitude coverage.

Five source pulls come from one driving session. More variations do not create more independent measured sessions. An event-held-out test assesses transfer to another event from that session, not to a different engine or driving session.

## Research sources and report

Iwana and Uchida (2021) compare time-series augmentation methods, including time warping, scaling and jittering. Effects depend on the model and dataset, and some transformations reduce performance. Wen et al. (2021) discuss temporal dependence, multivariate dynamics and task-specific validity.

These sources support studying augmentation, not the realism of a particular engine scenario. The current timestamp scaling is simpler than the surveyed warping implementations. Exact titles, publication details and links were checked on the publisher/proceedings pages. A later source review will assess relevance and the support for each report claim.

The final report should explain the decisions and results without requiring the repository. The repository can provide implementation, event tables and notes and cite the final commit or release.

Sources: ../reports/sources_2026-11-26.md.

## Next session

Define the first lambda/AFR scenario during a loaded part of a development pull. Specify its start, duration, magnitude, load condition and label rule before implementation.

Engine literature is still needed for the relevant mixture, load, timing and temperature mechanisms. No engine-risk paper was verified during this session, and no lambda conversion or risk labels have been implemented.

Then check the modified sensor relationships, recalculate affected features and compare the scenario with its source event.